In [4]:
!uv init


Initialized project `content`


In [ ]:
!uv add -r requirements.txt

In [ ]:
# Fix opentelemetry dependency for chromadb
# Uninstall potentially conflicting opentelemetry packages
!pip uninstall -y opentelemetry-api opentelemetry-sdk

# Install required libraries, allowing chromadb to manage its opentelemetry dependencies
!pip install sentence-transformers
!pip install chromadb
!pip install faiss-cpu

!pip install langchain_community
!pip install langchain
!pip install langchain_core
!pip install langchain_text_splitters


In [6]:
from langchain_core.documents import Document

In [12]:
doc= Document(
    page_content = "This is a text file. Foe demonstrate \"Document\" class inside Langchain",
    metadata={
        "source":"https:/examples.com/example.txt",
        "author":"John Doe",
        "date":"2023-01-01",
        "page_count":1
    }
    )

doc

Document(metadata={'source': 'https:/examples.com/example.txt', 'author': 'John Doe', 'date': '2023-01-01', 'page_count': 1}, page_content='This is a text file. Foe demonstrate "Document" class inside Langchain')

In [7]:
import os
os.makedirs("/content/data/text_files/",exist_ok=True)


### TextLoader

In [19]:
loader= TextLoader("/content/data/text_files/Future Improvements to the Agricult.txt",encoding="utf-8")
document= loader.load()
print(document)


[Document(metadata={'source': '/content/data/text_files/Future Improvements to the Agricult.txt'}, page_content='Future Improvements to the Agricultural Scoring Method Integrate Temporal Dynamics for Stage Detection \n\nCurrent stage boundaries rely on fixed NDVI thresholds, which may not capture local phenological variability. Future iterations should adopt dynamic, time-series–based stage detection, where growth stages are derived from NDVI or SAR backscatter peaks and valleys. This approach, as demonstrated by (McGiven & Müller, 2025), enables adaptive segmentation of the crop cycle based on real phenological transitions rather than static thresholds.\n\nFuse SAR-Derived Structural Indicators The existing scoring model is predominantly optical and may be affected by persistent cloud cover. Incorporating Sentinel-1 SAR polarization features—such as the Ratio Polarization Index (RPI), Normalized Difference Polarization Index (NDPI), and their temporal slope can improve robustness and 

#Directory Loader


##Text directory loader

In [20]:
from langchain_community.document_loaders import DirectoryLoader

In [26]:
dir_load= DirectoryLoader(
    "/content/data/text_files/",
    glob="**/*.txt",
    show_progress=True,
    use_multithreading=True,
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)

txt_files=dir_load.load()

txt_files


100%|██████████| 3/3 [00:00<00:00, 997.69it/s]


[Document(metadata={'source': '/content/data/text_files/Future Improvements to the Agricult.txt'}, page_content='Future Improvements to the Agricultural Scoring Method Integrate Temporal Dynamics for Stage Detection \n\nCurrent stage boundaries rely on fixed NDVI thresholds, which may not capture local phenological variability. Future iterations should adopt dynamic, time-series–based stage detection, where growth stages are derived from NDVI or SAR backscatter peaks and valleys. This approach, as demonstrated by (McGiven & Müller, 2025), enables adaptive segmentation of the crop cycle based on real phenological transitions rather than static thresholds.\n\nFuse SAR-Derived Structural Indicators The existing scoring model is predominantly optical and may be affected by persistent cloud cover. Incorporating Sentinel-1 SAR polarization features—such as the Ratio Polarization Index (RPI), Normalized Difference Polarization Index (NDPI), and their temporal slope can improve robustness and 

##PDF Directory Loader

In [17]:
#!uv add pymupdf
!pip install pymupdf

In [18]:
os.makedirs("/content/data/pdf_files",exist_ok=True)

In [22]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_dir_load= DirectoryLoader(
    "/content/data/pdf_files/",
    glob="**/*.pdf",
    show_progress=True,
    use_multithreading=True,
    loader_cls=PyMuPDFLoader
)

pdf_files=pdf_dir_load.load()

pdf_files

100%|██████████| 2/2 [00:00<00:00,  3.60it/s]


[Document(metadata={'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2025-12-23T03:36:48+00:00', 'source': '/content/data/pdf_files/Sentinel-2 Coverage Analysis.pdf', 'file_path': '/content/data/pdf_files/Sentinel-2 Coverage Analysis.pdf', 'total_pages': 7, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-23T03:36:48+00:00', 'trapped': '', 'modDate': "D:20251223033648+00'00'", 'creationDate': "D:20251223033648+00'00'", 'page': 0}, page_content='Sentinel-2 is a European Earth observation satellite constellation comprising two identical\nsatellites (Sentinel-2A and Sentinel-2B) operating as part of the Copernicus program. While\nthe mission provides global coverage, the frequency and quality of observations vary\nsignificantly across different geographical regions. This document explores why coverage\ndiffers globally and specifically compares developed nations (USA, Netherlands) with\ntropical regions (Cambodia, India

# Chunking

In [23]:
!uv add langchain-text-splitters

Resolved 142 packages in 868ms
Audited 138 packages in 71ms


In [27]:
import langchain
import langchain_core
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from pathlib import Path
os.makedirs("/content/data/pdf_files",exist_ok=True)

In [25]:
def process_all_pdfs(pdf_dir):
  """ Process all pdf files in a directory """
  all_documents=[]

  pdf_files=list(Path(pdf_dir).glob("**/*.pdf"))
  print(f"found {len(pdf_files)} pdf files inside directory")


  for pdf_file in pdf_files:
    print(f"Processing {pdf_file}")
    try:
        loader=PyMuPDFLoader(str(pdf_file))
        documents=loader.load()
        # add Source info to meta data
        for doc in documents:
          doc.metadata["source"]=str(pdf_file)
          doc.metadata["page"]=str(doc.metadata["page"])
          doc.metadata["file_type"]= "pdf"

        all_documents.extend(documents)
        print(f"loaded {len(documents)} pages")
    except Exception as e:
      print(f"Error processing {pdf_file}: {e}")
  print(f"Total documents: {len(all_documents)}")
  return all_documents


In [26]:
all_pdf_document= process_all_pdfs("/content/data/pdf_files")


found 2 pdf files inside directory
Processing /content/data/pdf_files/1a_Developing_a_Geodatabase_in_Support_of_Agricultural_Planning.pdf
loaded 39 pages
Processing /content/data/pdf_files/Sentinel-2 Coverage Analysis.pdf
loaded 7 pages
Total documents: 46


In [29]:
all_pdf_document

[Document(metadata={'producer': 'GPL Ghostscript 9.56.1', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': "D:20251114070112Z00'00'", 'source': '/content/data/pdf_files/AI-Agents.pdf', 'file_path': '/content/data/pdf_files/AI-Agents.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20251114070112Z00'00'", 'trapped': '', 'modDate': "D:20251114070112Z00'00'", 'creationDate': "D:20251114070112Z00'00'", 'page': '0', 'file_type': 'pdf'}, page_content='AI Agents\nIntelligent, action-oriented \nsystems.\nComplex Automation\nMoving beyond simple chatbots.\nEnterprise Solutions\nTransforming how organizations \noperate.\nGoogle Agent \nDevelopment Kit (ADK)\nUnlock complex automation within your enterprise using the ADK.'),
 Document(metadata={'producer': 'GPL Ghostscript 9.56.1', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': "D:20251114070112Z00'00'", 'source': '/content/da

In [28]:
## Text splitiing and chuking

def split_documents(documents,chunk_size=500,chunk_overlap=100):
  """ Slipt document into smaller chunks for better RAG performance"""
  print("="*50,"Chunking START","="*50)
  text_splitter= RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      length_function=len,
      separators=["\n\n", "\n"," ", ""]
  )
  split_documents= text_splitter.split_documents(documents)
  print(f"Original document: {len(documents)}, Slipt into {len(split_documents)} chunks")

  if split_documents:
    print(f"Example chunk: {split_documents[0].page_content}")

  print("="*50,"Chunking Done","="*50)
  return split_documents


In [29]:
chunk_docs= split_documents(all_pdf_document)

================================================== Chunking START ==================================================
Original document: 46, Slipt into 74 chunks
Example chunk: Developing a 
geodatabase in support 
of agricultural planning
& Use Cases
National Training and Workshops 
Phnom Penn, Cambodia May 8-9
Hanoi, Vietnan May 13-14 2025
================================================== Chunking Done ==================================================


In [ ]:
chunk_docs

Chunking Done

# Vector Storage

#embeding and Vector store

In [5]:
!pip install -U sentence-transformers



In [6]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 6.76569194e-02  6.34959415e-02  4.87130806e-02  7.93049484e-02
   3.74481045e-02  2.65281182e-03  3.93749401e-02 -7.09844474e-03
   5.93614615e-02  3.15370038e-02  6.00980595e-02 -5.29052690e-02
   4.06068154e-02 -2.59308740e-02  2.98427939e-02  1.12690637e-03
   7.35148638e-02 -5.03818318e-02 -1.22386597e-01  2.37028114e-02
   2.97265425e-02  4.24768664e-02  2.56337449e-02  1.99515442e-03
  -5.69190457e-02 -2.71598250e-02 -3.29035148e-02  6.60249218e-02
   1.19007207e-01 -4.58791330e-02 -7.26214126e-02 -3.25840451e-02
   5.23412861e-02  4.50553298e-02  8.25300720e-03  3.67024392e-02
  -1.39415609e-02  6.53918684e-02 -2.64272150e-02  2.06383236e-04
  -1.36643508e-02 -3.62810828e-02 -1.95043813e-02 -2.89738141e-02
   3.94270159e-02 -8.84090737e-02  2.62426236e-03  1.36713963e-02
   4.83063012e-02 -3.11566219e-02 -1.17329165e-01 -5.11690676e-02
  -8.85288119e-02 -2.18963195e-02  1.42986281e-02  4.44167927e-02
  -1.34815322e-02  7.43392557e-02  2.66382676e-02 -1.98762883e-02
   1.79191

In [12]:
import numpy as np
from typing import List, Tuple, Dict, Any
import uuid
from sklearn.metrics.pairwise import cosine_similarity
from chromadb.config import Settings
import chromadb
from langchain_community.document_loaders import TextLoader
import os

In [7]:
class embeddingManager:
  """Handels document embedding generation using SentenceTransformer """
  def __init__(self,model_name:str="all-MiniLM-L6-v2"):
    """
    Intializer for embedingg manager

    Args:
      model_name (str): SentenceTransformer model name, Hugging face model name for sentence embeding
    """
    self.model_name= model_name
    self.model= None
    self._load_model()
  def _load_model(self):
    """ Load SentenceTransformer model """
    try:
      from sentence_transformers import SentenceTransformer
      self.model= SentenceTransformer(self.model_name)
      print(f"Model load successfully, embedding diamension is {self.model.get_sentence_embedding_dimension()}")
    except Exception as e:
      print(f"Error loading model {self.model_name}: {e}")
      raise
  def generate_embedding(self,texts:List[str]) -> np.ndarray:
    """
    generate embedding for a list of texts.

    Args:
       texts: List of string to embed
    returns:
       numpy array of embedding with shape (len(texts), embedding_dim)
    """
    if not self.model:
      raise ValueError("Model not loaded")
    print(f"Generating embbeding for {len(texts)} texts....")
    embeddings= self.model.encode(texts,show_progress_bar=True)
    print(f"generated embeding for shape{embeddings.shape}")
    return embeddings



In [8]:
## Intialize the embeding manager
embeding_manager= embeddingManager()
embeding_manager

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model load successfully, embedding diamension is 384


#Vectore Store

In [52]:
class Vector_Store:
  """
  Manages documents vector store using ChromaDB
  """

  def __init__(self,collection_name:str="pdf_documents", persist_dir:str='/content/data/pdf_vectore'):
    """
    Intializer of the vectore store
    Args:
      collection_name:name of the ChromaDB collection
      persist_dir:Directory to keep the vectore store data
    """
    self.collection_name= collection_name
    self.persist_dir= persist_dir
    self.client= None
    self.collection= None
    self._initilize_store()

  def _initilize_store(self):
    """ Initialize the vectore store """
    try:
        os.makedirs(self.persist_dir,exist_ok=True)
        self.client= chromadb.PersistentClient(path=self.persist_dir)

        #Get or create collection
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "Pdf data embedding vector for RAG"}

        )

        print(f"vectore store initilized successfully")
        print(f"Collection name: {self.collection.name}")
        print(f"Collection metadata: {self.collection.metadata}")
        print(f"Collection Count:{self.collection.count()}")
    except Exception as e:
      print(f"Error initilizing vectore store: {e}")
      raise

  def add_document(self, documents:List[Any],embbedings:np.ndarray):
    """
    Add documents to the vectore store
    Args:
      documents: List of documents to add to the vectore store
      embedding: Coresponding embeding for the documents

    """
    print(f"DEBUG: Length of documents received: {len(documents)}")
    print(f"DEBUG: Length of embeddings received: {len(embbedings)}")
    if len(documents) != len(embbedings):
      raise ValueError("Number of documents and embeddings must be equal")


    print(f"Adding {len(documents)} documents to the vectore store")


    ids=[]
    metadatas=[]
    documents_texts=[]
    embeding_list=[]
    for i,(doc,embedding) in enumerate(zip(documents,embbedings)):
      docid= f"doc_{uuid.uuid4().hex[:8]}_{i}"
      ids.append(docid)
      metadata = dict(doc.metadata)
      metadata["doc_index"]=i
      metadata["content_length"]=len(doc.page_content)
      metadatas.append(metadata)
      documents_texts.append(doc.page_content)
      embeding_list.append(embedding.tolist())

    print(f"Adding {len(documents)} documents to the vectore store")

    try:
      self.collection.add(
          ids=ids,
          documents=documents_texts,
          metadatas=metadatas,
          embeddings=embeding_list
      )
      print(f"Added {len(documents)} documents to the vectore store")

      print(f"Collection Count: {self.collection.count()}")
    except Exception as e:
        print(f"Error adding documents to the vectore store: {e}")
        raise

In [53]:
vector_store= Vector_Store()
vector_store

vectore store initilized successfully
Collection name: pdf_documents
Collection metadata: {'description': 'Pdf data embedding vector for RAG'}
Collection Count:0


In [39]:
## Convert chunk text to embeding

texts= [doc.page_content for doc in chunk_docs]


In [ ]:
## generate embedings
embeding_texts= embeding_manager.generate_embedding(texts)


In [54]:

# store it in chromadb
vector_store.add_document(chunk_docs,embeding_texts)

DEBUG: Length of documents received: 74
DEBUG: Length of embeddings received: 74
Adding 74 documents to the vectore store
Adding 74 documents to the vectore store
Added 74 documents to the vectore store
Collection Count: 74


#Full Data Pipeline : PDF -> Chunks-> Embeding -> vectorDB(ChromaDB)

In [55]:
# Fix opentelemetry dependency for chromadb
# Uninstall potentially conflicting opentelemetry packages
!pip uninstall -y opentelemetry-api opentelemetry-sdk

# Install required libraries, allowing chromadb to manage its opentelemetry dependencies
!pip install sentence-transformers
!pip install chromadb
!pip install faiss-cpu

!pip install langchain_community
!pip install langchain
!pip install langchain_core
!pip install langchain_text_splitters


!pip install pymupdf

Found existing installation: opentelemetry-api 1.40.0
Uninstalling opentelemetry-api-1.40.0:
  Successfully uninstalled opentelemetry-api-1.40.0
Found existing installation: opentelemetry-sdk 1.40.0
Uninstalling opentelemetry-sdk-1.40.0:
  Successfully uninstalled opentelemetry-sdk-1.40.0
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
Using cached opentelemetry_api-1.40.0-py3-none-any.whl (68 kB)
Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl (141 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but 

## 0. Load Pdf files

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_dir_load= DirectoryLoader(
    "/content/data/pdf_files/",
    glob="**/*.pdf",
    show_progress=True,
    use_multithreading=True,
    loader_cls=PyMuPDFLoader
)

pdf_files=pdf_dir_load.load()

pdf_files

##1. Process PDF Files


In [ ]:
def process_all_pdfs(pdf_dir):
  """ Process all pdf files in a directory """
  all_documents=[]

  pdf_files=list(Path(pdf_dir).glob("**/*.pdf"))
  print(f"found {len(pdf_files)} pdf files inside directory")


  for pdf_file in pdf_files:
    print(f"Processing {pdf_file}")
    try:
        loader=PyMuPDFLoader(str(pdf_file))
        documents=loader.load()
        # add Source info to meta data
        for doc in documents:
          doc.metadata["source"]=str(pdf_file)
          doc.metadata["page"]=str(doc.metadata["page"])
          doc.metadata["file_type"]= "pdf"

        all_documents.extend(documents)
        print(f"loaded {len(documents)} pages")
    except Exception as e:
      print(f"Error processing {pdf_file}: {e}")
  print(f"Total documents: {len(all_documents)}")
  return all_documents


##2. Chunking

In [ ]:
## Text splitiing and chuking

def split_documents(documents,chunk_size=500,chunk_overlap=100):
  """ Slipt document into smaller chunks for better RAG performance"""
  print("="*50,"Chunking START","="*50)
  text_splitter= RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap,
      length_function=len,
      separators=["\n\n", "\n"," ", ""]
  )
  split_documents= text_splitter.split_documents(documents)
  print(f"Original document: {len(documents)}, Slipt into {len(split_documents)} chunks")

  if split_documents:
    print(f"Example chunk: {split_documents[0].page_content}")

  print("="*50,"Chunking Done","="*50)
  return split_documents




## 3. Embeding Manager Class

In [ ]:
import numpy as np
from typing import List, Tuple, Dict, Any
import uuid
from sklearn.metrics.pairwise import cosine_similarity
from chromadb.config import Settings
import chromadb
from langchain_community.document_loaders import TextLoader
import os

class embeddingManager:
  """Handels document embedding generation using SentenceTransformer """
  def __init__(self,model_name:str="all-MiniLM-L6-v2"):
    """
    Intializer for embedingg manager

    Args:
      model_name (str): SentenceTransformer model name, Hugging face model name for sentence embeding
    """
    self.model_name= model_name
    self.model= None
    self._load_model()
  def _load_model(self):
    """ Load SentenceTransformer model """
    try:
      from sentence_transformers import SentenceTransformer
      self.model= SentenceTransformer(self.model_name)
      print(f"Model load successfully, embedding diamension is {self.model.get_sentence_embedding_dimension()}")
    except Exception as e:
      print(f"Error loading model {self.model_name}: {e}")
      raise
  def generate_embedding(self,texts:List[str]) -> np.ndarray:
    """
    generate embedding for a list of texts.

    Args:
       texts: List of string to embed
    returns:
       numpy array of embedding with shape (len(texts), embedding_dim)
    """
    if not self.model:
      raise ValueError("Model not loaded")
    print(f"Generating embbeding for {len(texts)} texts....")
    embeddings= self.model.encode(texts,show_progress_bar=True)
    print(f"generated embeding for shape{embeddings.shape}")
    return embeddings



##4. Vector store class

In [ ]:
class Vector_Store:
  """
  Manages documents vector store using ChromaDB
  """

  def __init__(self,collection_name:str="pdf_documents", persist_dir:str='/content/data/pdf_vectore'):
    """
    Intializer of the vectore store
    Args:
      collection_name:name of the ChromaDB collection
      persist_dir:Directory to keep the vectore store data
    """
    self.collection_name= collection_name
    self.persist_dir= persist_dir
    self.client= None
    self.collection= None
    self._initilize_store()

  def _initilize_store(self):
    """ Initialize the vectore store """
    try:
        os.makedirs(self.persist_dir,exist_ok=True)
        self.client= chromadb.PersistentClient(path=self.persist_dir)

        #Get or create collection
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "Pdf data embedding vector for RAG"}

        )

        print(f"vectore store initilized successfully")
        print(f"Collection name: {self.collection.name}")
        print(f"Collection metadata: {self.collection.metadata}")
        print(f"Collection Count:{self.collection.count()}")
    except Exception as e:
      print(f"Error initilizing vectore store: {e}")
      raise

  def add_document(self, documents:List[Any],embbedings:np.ndarray):
    """
    Add documents to the vectore store
    Args:
      documents: List of documents to add to the vectore store
      embedding: Coresponding embeding for the documents

    """
    print(f"DEBUG: Length of documents received: {len(documents)}")
    print(f"DEBUG: Length of embeddings received: {len(embbedings)}")
    if len(documents) != len(embbedings):
      raise ValueError("Number of documents and embeddings must be equal")


    print(f"Adding {len(documents)} documents to the vectore store")


    ids=[]
    metadatas=[]
    documents_texts=[]
    embeding_list=[]
    for i,(doc,embedding) in enumerate(zip(documents,embbedings)):
      docid= f"doc_{uuid.uuid4().hex[:8]}_{i}"
      ids.append(docid)
      metadata = dict(doc.metadata)
      metadata["doc_index"]=i
      metadata["content_length"]=len(doc.page_content)
      metadatas.append(metadata)
      documents_texts.append(doc.page_content)
      embeding_list.append(embedding.tolist())

    print(f"Adding {len(documents)} documents to the vectore store")

    try:
      self.collection.add(
          ids=ids,
          documents=documents_texts,
          metadatas=metadatas,
          embeddings=embeding_list
      )
      print(f"Added {len(documents)} documents to the vectore store")

      print(f"Collection Count: {self.collection.count()}")
    except Exception as e:
        print(f"Error adding documents to the vectore store: {e}")
        raise

## 5. Integration

In [ ]:

#Load Pdf files
all_pdf_document= process_all_pdfs("/content/data/pdf_files")

#chunk Pdf files
chunk_docs= split_documents(all_pdf_document)

#convert chunk to text
texts= [doc.page_content for doc in chunk_docs]

# embedding the chunk texts
embeding_manager= embeddingManager()
embeding_texts= embeding_manager.generate_embedding(texts)

# store in a vectore database
vector_store= Vector_Store()
vector_store.add_document(chunk_docs,embeding_texts)


##6. RAG- Retriver


*  Search about specific topic in our vector store.



In [60]:
class RAG_Retriver:
  """
  Handel query based retrival from out vector store
  """
  def __init__(self,vector_store:Vector_Store, embeding_manager:embeddingManager):
    """
    Intializer for RAG retriver
    Args:
      vectore_stopre: Vector_Store object
      embeding_manager: embeddingManager object


    """


    self.vector_store= vector_store
    self.embeding_manager= embeding_manager

  def retriver(self,query:str,top_k:int=5,score_threshold:float=0.0):
    """
    Retrieve relevant document for a query
    Args:
      query: query string to search for
      top_k: number of documents to return
      score_threshold: minimum score to return
    Returns:
      List of dictionary with document metadata and score
    """
    print(f"Retriving documents for query: {query}")
    print(f"Top k: {top_k}, Score threshold: {score_threshold}")

    query_embeding= self.embeding_manager.generate_embedding([query])[0]
    try:

      results= self.vector_store.collection.query(
          query_embeddings= [query_embeding.tolist()],
          n_results= top_k
      )

      retrived_docs=[]
      if results['documents'] and results['documents'][0]:
        documents= results['documents'][0]
        distances= results['distances'][0]
        metadata= results['metadatas'][0]
        ids= results['ids'][0]
        for i,(doc_id,document,distance,metadata) in enumerate(zip(ids,documents,distances,metadata)):
          similarity_score= 1- distance
          if similarity_score > score_threshold:
            retrived_docs.append({
                "id": doc_id,
                "content": document,
                "metadata": metadata,
                "score": similarity_score,
                "distance": distance,
                "rank":i+1
            })
            print(f"Retrived Document:{len(retrived_docs)}")
          else:
            print(f"No documents found with score above threshold {score_threshold}")

        return retrived_docs

    except Exception as e:
        print(f"Error retriving documents: {e}")
        return []

In [62]:
rag_retriver= RAG_Retriver(vector_store,embeding_manager)

In [71]:
rag_retriver.retriver("Practical Implications for Agriculture")

Retriving documents for query: Practical Implications for Agriculture
Top k: 5, Score threshold: 0.0
Generating embbeding for 1 texts....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

generated embeding for shape(1, 384)
Retrived Document:1
Retrived Document:2
No documents found with score above threshold 0.0
No documents found with score above threshold 0.0
No documents found with score above threshold 0.0


[{'id': 'doc_298351e1_6',
  'content': 'Next steps, \nIdeas for collaborations\n• Enhance country transfer of this technology, to use local data for \nimproving and expanding use cases. Possible themes: sustainability, \ncarbon credits, crop insurance, fertilizer subsidies, climate resilience\n• Use foundational AI models and advanced computing power to \nautomatically detect crops and perform useful yield analysis (next \ntalk)',
  'metadata': {'page': '5',
   'format': 'PDF 1.7',
   'doc_index': 6,
   'creationdate': '2025-05-09T10:18:41+07:00',
   'content_length': 385,
   'producer': 'Microsoft® PowerPoint® for Microsoft 365',
   'file_type': 'pdf',
   'author': 'Taussi, Anna (OIND)',
   'title': '',
   'trapped': '',
   'file_path': '/content/data/pdf_files/1a_Developing_a_Geodatabase_in_Support_of_Agricultural_Planning.pdf',
   'creationDate': "D:20250509101841+07'00'",
   'creator': 'Microsoft® PowerPoint® for Microsoft 365',
   'modDate': "D:20250509101841+07'00'",
   'total_pa

## Integration with groq LLM

In [73]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.0 MB/s eta 0:00:00


In [72]:
from google.colab import userdata
groq_api_key= userdata.get('groq_api_key')

In [75]:
import os
from langchain_groq import ChatGroq


In [91]:
model=ChatGroq(groq_api_key=groq_api_key,model_name='llama-3.1-8b-instant',temperature=0.1, max_tokens=1024)

In [96]:
def simple_rag(query,retriver,llm,top_k=3):
  results= retriver.retriver(query,top_k=top_k)
  context="\n\n".join([doc['content'] for  doc in results]) if results else ""
  if not context:
    return "No documents found"
  prompt= f""" use the follwing context to anser the user questions concisely, merge every thing and give a summaerized answer of the asked question
      Context: {context}

      Questions:{query}
      Answer:
  """
  response= llm.invoke([prompt.format(context=context,query=query)])
  return response.content


In [97]:
answer = simple_rag("Cambodia vs India coverage of Sentinal-2",rag_retriver,model,top_k=3)
print(answer)

Retriving documents for query: Cambodia vs India coverage of Sentinal-2
Top k: 3, Score threshold: 0.0
Generating embbeding for 1 texts....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

generated embeding for shape(1, 384)
Retrived Document:1
Retrived Document:2
Retrived Document:3
**Comparison of Cambodia and India's Sentinel-2 Coverage**

Cambodia experiences the poorest coverage due to its equatorial position and severe monsoon cloud cover (80-90% during the monsoon season). In contrast, India has variable coverage, with northern regions approaching temperate region quality and southern regions experiencing tropical limitations.

**Key differences:**

- **Cloud Cover:** Cambodia: 80-90% (monsoon season), India: variable
- **Orbital Overlap:** Cambodia: minimal, India: more overlap
- **Effective Revisit:** Cambodia: not specified, India: 2-3 days

**Summary:** Cambodia's equatorial position and severe monsoon cloud cover result in the poorest Sentinel-2 coverage, while India's variable coverage is influenced by its geographical location and orbital overlap.
